# MD-CLARA: comparing IDCD, CAT, and DAT on patent cities

This notebook runs **all three multidomain dissimilarity strategies** on the same city-level patent data and compares their cluster solutions.

**Pairfam (individuals, activity + family):** [md_clara_pairfam_strategies.ipynb](md_clara_pairfam_strategies.ipynb).

| Strategy | What it measures |
|----------|------------------|
| **IDCD** | Dissimilarity on **observed joint** states (domain1+domain2+domain3) |
| **CAT** | Optimal matching with **additive** multidomain substitution/indel costs |
| **DAT** | **Sum** (here) of domain-specific OM distances |

**Prerequisite:** If you are new to MD-CLARA or this dataset, start with [md_clara_patent_cities_tutorial.ipynb](md_clara_patent_cities_tutorial.ipynb) (data loading, `SequenceData`, reading `MDClaraResult`).

We use the **same** `R`, `sample_size`, `kvals`, and `random_state` for every strategy so differences reflect the dissimilarity definition, not the CLARA random design.

## 1. Load aligned multidomain data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score

from sequenzo import SequenceData
from sequenzo.multidomain.clara import (
    md_clara,
    plot_md_clara_quality,
    plot_md_clara_stability,
)

%matplotlib inline

DATA_ROOT = Path(
    "/Users/lei/Documents/Sequenzo_all_folders/global-south-sequential-patent-data"
    "/data/02_analysis_ready/sequence/domain"
)

PATHS = {
    "computing": DATA_ROOT
    / "computing/y3/computing_3years_global_quintiles_patent_sequence_data_by_city.csv",
    "medicine": DATA_ROOT
    / "medicine/y3/medicine_3years_global_quintiles_patent_sequence_data_by_city.csv",
    "environmental": DATA_ROOT
    / "environmental_technology/y3"
    / "environmental_technology_3years_global_quintiles_patent_sequence_data_by_city.csv",
}

dfs = {name: pd.read_csv(path) for name, path in PATHS.items()}

meta_cols = ["city_id", "city", "country_code", "country_name", "continent"]
time_cols = [c for c in dfs["computing"].columns if c[0].isdigit() and "-" in c]
STATES = ["No patents", "Very Low", "Low", "Medium", "High", "Very High"]

common_ids = set(dfs["computing"]["city_id"])
for name in ("medicine", "environmental"):
    common_ids &= set(dfs[name]["city_id"])

aligned = {}
for name, df in dfs.items():
    aligned[name] = (
        df[df["city_id"].isin(common_ids)]
        .sort_values("city_id")
        .reset_index(drop=True)
    )

city_meta = aligned["computing"][meta_cols].copy()
domains = [
    SequenceData(
        aligned[name],
        time=time_cols,
        id_col="city_id",
        states=STATES,
        labels=STATES,
    )
    for name in ("computing", "medicine", "environmental")
]

print(f"Cities: {len(city_meta)}  |  Time points: {len(time_cols)}")

## 2. Shared CLARA settings and strategy-specific distances

All strategies use optimal matching (`OM`) with a constant substitution matrix and `indel=1`.  
Average distances (`avg_dist`) are **not comparable across strategies** (different distance scales); compare **patterns across k** within each strategy, and use **partition agreement** (ARI) across strategies at a fixed `k`.

In [ ]:
N_DOM = len(domains)

CLARA_KWARGS = dict(
    R=30,
    sample_size=200,
    kvals=[2, 3, 4, 5, 6],
    criteria=("distance",),
    stability=True,
    random_state=42,
    n_jobs=-1,
    verbose=True,
)

STRATEGY_CONFIG = {
    "idcd": {
        "label": "IDCD (joint observed states)",
        "distance_params": {
            "method": "OM",
            "sm": "CONSTANT",
            "indel": 1,
            "norm": "none",
        },
    },
    "cat": {
        "label": "CAT (additive multidomain costs)",
        "distance_params": {
            "method": "OM",
            "sm": ["CONSTANT"] * N_DOM,
            "indel": 1,
            "norm": "none",
        },
    },
    "dat": {
        "label": "DAT (sum of domain distances)",
        "distance_params": {
            "method_params": [
                {"method": "OM", "sm": "CONSTANT", "indel": 1},
            ]
            * N_DOM,
            "domain_weights": [1.0] * N_DOM,
            "link": "sum",
        },
    },
}

## 3. Run MD-CLARA for each strategy

This cell runs three analyses in sequence (IDCD → CAT → DAT). Expect several minutes on a laptop with the settings above.

In [ ]:
results = {}

for strategy, cfg in STRATEGY_CONFIG.items():
    print("\n" + "=" * 60)
    print(cfg["label"])
    print("=" * 60)
    results[strategy] = md_clara(
        domains,
        strategy=strategy,
        distance_params=cfg["distance_params"],
        **CLARA_KWARGS,
    )

list(results.keys())

## 4. Compare cluster quality statistics

In [ ]:
quality_rows = []
for strategy, res in results.items():
    stats = res.stats.copy()
    stats["strategy"] = strategy
    quality_rows.append(stats)

quality = pd.concat(quality_rows, ignore_index=True)
quality[["strategy", "k", "avg_dist", "db", "xb", "ari08", "jc08"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for strategy in STRATEGY_CONFIG:
    sub = quality[quality["strategy"] == strategy]
    ax.plot(
        sub["k"],
        sub["avg_dist"],
        marker="o",
        label=STRATEGY_CONFIG[strategy]["label"],
    )
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Average distance to medoids")
ax.set_title("Within-strategy fit by k (scales differ across strategies)")
ax.legend(loc="best", fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Quality and stability plots (per strategy)

In [ ]:
for strategy, res in results.items():
    title = STRATEGY_CONFIG[strategy]["label"]
    plot_md_clara_quality(res, title=f"{title} — cluster quality")
    plt.show()
    plot_md_clara_stability(res, title=f"{title} — stability")
    plt.show()

## 6. Compare cluster partitions at a chosen k

Pick one `k` for substantive interpretation, then check whether strategies assign cities to similar groups (**adjusted Rand index**, ARI ∈ [−1, 1]; 1 = identical partitions).

In [ ]:
K_CHOSEN = 4

labels = {
    strategy: results[strategy].best_clustering(K_CHOSEN)
    for strategy in STRATEGY_CONFIG
}

strategies = list(STRATEGY_CONFIG.keys())
ari_matrix = pd.DataFrame(index=strategies, columns=strategies, dtype=float)

for a in strategies:
    for b in strategies:
        ari_matrix.loc[a, b] = adjusted_rand_score(labels[a], labels[b])

print(f"Adjusted Rand index between strategies at k = {K_CHOSEN}")
ari_matrix.round(3)

In [ ]:
compare = city_meta.copy()
for strategy in strategies:
    compare[f"cluster_{strategy}_k{K_CHOSEN}"] = labels[strategy]

pd.crosstab(
    compare[f"cluster_idcd_k{K_CHOSEN}"],
    compare[f"cluster_cat_k{K_CHOSEN}"],
    rownames=["IDCD"],
    colnames=["CAT"],
)

In [ ]:
pd.crosstab(
    compare[f"cluster_idcd_k{K_CHOSEN}"],
    compare[f"cluster_dat_k{K_CHOSEN}"],
    rownames=["IDCD"],
    colnames=["DAT"],
)

## 7. Export combined cluster assignments

In [ ]:
export = compare.copy()
for strategy, res in results.items():
    export = export.join(res.clustering.add_prefix(f"{strategy}_"))

out_path = Path("md_clara_patent_cities_all_strategies.csv")
export.to_csv(out_path, index=False)
print(f"Saved: {out_path.resolve()}")

## 8. How to choose a strategy?

- **IDCD** — Default when domains are **associated** (joint states matter). Uses only **observed** combined states; good for substantive multidomain typologies when co-occurrence patterns are strong (as with patent quintiles across related tech fields).
- **CAT** — Useful when you want OM on a **concatenated** multidomain representation with explicit additive costs; can behave differently when many joint states are rare.
- **DAT** — Treats domains more **separately** (here: sum of three OM matrices). Often more robust when association is weak, but can underplay joint dynamics if domains are strongly linked.

**Practical workflow:** Run all three (this notebook), compare stability and ARI at your preferred `k`, then deepen interpretation with domain-specific sequence plots (`plot_md_cluster_by_domain`) on the strategy you adopt.

See also: [main multidomain tutorial](main_tutorial.ipynb), `docs/multidomain/md-clara-guidelines.md`.